In [1]:
!pip install -q xgboost

In [2]:
from google.colab import files
uploaded = files.upload()

Saving train.csv to train.csv


In [3]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

data = pd.read_csv("train.csv")
data.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [4]:
y = data["SalePrice"]

features = [
    "LotArea",
    "YearBuilt",
    "1stFlrSF",
    "2ndFlrSF",
    "FullBath",
    "BedroomAbvGr",
    "Neighborhood",
    "HouseStyle"
]

X = data[features]

numeric_features = [
    "LotArea",
    "YearBuilt",
    "1stFlrSF",
    "2ndFlrSF",
    "FullBath",
    "BedroomAbvGr"
]

categorical_features = [
    "Neighborhood",
    "HouseStyle"
]

In [5]:
numeric_transformer = SimpleImputer(strategy="median")

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [6]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=0
)

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", rf_model)
])

rf_scores = -cross_val_score(
    rf_pipeline,
    X,
    y,
    cv=5,
    scoring="neg_mean_absolute_error"
)

print("Random Forest - MAE por fold:", rf_scores)
print("Random Forest - MAE medio:", rf_scores.mean())

Random Forest - MAE por fold: [21392.1855773  21899.08272505 21926.48897489 20706.91950913
 24020.34527805]
Random Forest - MAE medio: 21989.004412883234


In [7]:
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=0,
    objective="reg:squarederror"
)

xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

xgb_scores = -cross_val_score(
    xgb_pipeline,
    X,
    y,
    cv=5,
    scoring="neg_mean_absolute_error"
)

print("XGBoost - MAE por fold:", xgb_scores)
print("XGBoost - MAE medio:", xgb_scores.mean())

XGBoost - MAE por fold: [20777.14453125 21578.22070312 21623.73632812 20413.109375
 22506.71484375]
XGBoost - MAE medio: 21379.78515625


In [8]:
print("----- COMPARACIÓN FINAL -----")
print("Random Forest MAE medio:", rf_scores.mean())
print("XGBoost MAE medio:", xgb_scores.mean())

if rf_scores.mean() < xgb_scores.mean():
    print("Mejor modelo: Random Forest")
else:
    print("Mejor modelo: XGBoost")

----- COMPARACIÓN FINAL -----
Random Forest MAE medio: 21989.004412883234
XGBoost MAE medio: 21379.78515625
Mejor modelo: XGBoost
